# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

try:
    load_dotenv(".env")
except PermissionError as error:
    raise PermissionError(
        "configuration.load_dotenv | PermissionError: "
        f"{error} | Context: .env file | "
        "Suggestion: check file permissions and rerun the configuration cell."
    ) from error
except OSError as error:
    raise OSError(
        "configuration.load_dotenv | OSError: "
        f"{error} | Context: .env file | "
        "Suggestion: verify the .env path and filesystem access."
    ) from error

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
# Keep credentials in .env; never commit them to the notebook.
EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

In [3]:
# Set environment variables only when a key is configured.
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Pydantic Models for Validation

In [4]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

## 4. Reusable Helpers

In [5]:
import json
import re
from typing import Any


def format_error(function_name: str, error_type: str, message: str, context: str, suggestion: str) -> str:
    """Create a consistent, actionable error message."""
    return (
        f"{function_name} | {error_type}: {message} | "
        f"Context: {context} | Suggestion: {suggestion}"
    )


def require_configured_setting(value: str | None, name: str, source: str) -> str:
    """Return a configured setting or raise an actionable configuration error."""
    if not value:
        raise EnvironmentError(format_error(
            "require_configured_setting",
            "EnvironmentError",
            f"missing {name}",
            source,
            f"Add {name}=... to {source} and restart the notebook.",
        ))
    return value


def load_json_file(file_path: str) -> Any:
    """Load JSON test/configuration data with actionable file errors."""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError as error:
        raise FileNotFoundError(format_error(
            "load_json_file",
            "FileNotFoundError",
            str(error),
            f"file path {file_path}",
            "Check the path and make sure the JSON file exists.",
        )) from error
    except PermissionError as error:
        raise PermissionError(format_error(
            "load_json_file",
            "PermissionError",
            str(error),
            f"file path {file_path}",
            "Check read permissions for the JSON file.",
        )) from error
    except json.JSONDecodeError as error:
        raise json.JSONDecodeError(
            format_error(
                "load_json_file",
                "JSONDecodeError",
                error.msg,
                f"file path {file_path}, line {error.lineno}, column {error.colno}",
                "Fix the JSON syntax at the reported line and column.",
            ),
            error.doc,
            error.pos,
        ) from error


def normalize_optional_text(value: str | None, default: str) -> str:
    """Return trimmed text, or a default for blank optional input."""
    return value.strip() if isinstance(value, str) and value.strip() else default


def extract_json_payload(content: str) -> str:
    """Extract a JSON object/array from plain or fenced model output."""
    if not isinstance(content, str) or not content.strip():
        raise ValueError(format_error(
            "extract_json_payload",
            "ValueError",
            "model content is empty",
            "LLM response body",
            "Check the API response and ensure the model returned JSON text.",
        ))

    fenced_blocks = re.findall(
        r"\x60\x60\x60(?:json)?\s*(.*?)\s*\x60\x60\x60",
        content,
        flags=re.IGNORECASE | re.DOTALL,
    )
    candidates = fenced_blocks or [content]
    decoder = json.JSONDecoder()
    last_error = None

    for candidate in candidates:
        for match in re.finditer(r"[\[{]", candidate):
            try:
                _, end = decoder.raw_decode(candidate, match.start())
                return candidate[match.start():end]
            except json.JSONDecodeError as error:
                last_error = error

    if last_error:
        raise json.JSONDecodeError(
            format_error(
                "extract_json_payload",
                "JSONDecodeError",
                last_error.msg,
                f"response position {last_error.pos}",
                "Return a valid JSON object, optionally inside a complete code fence.",
            ),
            content,
            last_error.pos,
        )
    raise ValueError(format_error(
        "extract_json_payload",
        "ValueError",
        "no JSON object or array found",
        "LLM response body",
        "Check that the model output contains a JSON object.",
    ))


def parse_json_response(content: str) -> dict:
    """Parse model output after extracting its JSON payload."""
    payload = extract_json_payload(content)
    parsed = json.loads(payload)
    if not isinstance(parsed, dict):
        raise ValueError(format_error(
            "parse_json_response",
            "ValueError",
            "expected a JSON object",
            "parsed response root",
            "Ask the model for a JSON object matching the meal-plan schema.",
        ))
    required_fields = {"dietary_preference", "recipes", "shopping_list"}
    missing_fields = sorted(required_fields - parsed.keys())
    if missing_fields:
        raise KeyError(format_error(
            "parse_json_response",
            "KeyError",
            f"missing required fields: {', '.join(missing_fields)}",
            "meal-plan response object",
            "Check the model output against the MealPlan schema.",
        ))
    return parsed


def validate_meal_plan(data: dict) -> MealPlan:
    """Validate raw plan data with the single source of truth: Pydantic."""
    return MealPlan.model_validate(data)


def format_ingredients(ingredients: list[dict]) -> str:
    """Format ingredient dictionaries consistently across all outputs."""
    formatted = []
    for index, ingredient in enumerate(ingredients):
        try:
            formatted.append(f"{ingredient['quantity']} {ingredient['name']}")
        except KeyError as error:
            missing_field = error.args[0]
            raise KeyError(format_error(
                "format_ingredients",
                "KeyError",
                f"missing ingredient field '{missing_field}'",
                f"ingredient index {index}",
                "Provide both name and quantity for every ingredient.",
            )) from error
    return ", ".join(formatted)

### Helper function checks

In [6]:
# These checks use local sample data only; no OpenAI or email call is made.
assert normalize_optional_text(None, "random") == "random"
assert normalize_optional_text("  vegan  ", "omnivore") == "vegan"

plain_response = "Here is your plan:\n{\"dietary_preference\": \"vegan\"}"
assert extract_json_payload(plain_response) == "{\"dietary_preference\": \"vegan\"}"
fenced_response = "\x60\x60\x60json\n{\"dietary_preference\": \"vegan\", \"recipes\": [], \"shopping_list\": []}\n\x60\x60\x60"
assert parse_json_response(fenced_response)["dietary_preference"] == "vegan"

sample_ingredients = [
    {"quantity": "2 cups", "name": "rice"},
    {"quantity": "1 tbsp", "name": "soy sauce"},
]
assert format_ingredients(sample_ingredients) == "2 cups rice, 1 tbsp soy sauce"

sample_recipe = {
    "day": "Monday",
    "theme": "Italian",
    "dish_name": "Pasta",
    "ingredients": [
        {"name": f"ingredient-{index}", "quantity": "1"}
        for index in range(INGREDIENTS_PER_RECIPE)
    ],
    "instructions": "Cook and serve.",
}
sample_plan = {
    "dietary_preference": "vegan",
    "recipes": [
        {**sample_recipe, "day": day, "theme": f"Theme-{index}"}
        for index, day in enumerate(DAYS)
    ],
    "shopping_list": [],
}
assert validate_meal_plan(sample_plan).recipes[0].dish_name == "Pasta"

try:
    parse_json_response("Model reply: {not valid json}")
except json.JSONDecodeError as error:
    assert error.lineno == 1
else:
    raise AssertionError("Malformed JSON should raise JSONDecodeError")

invalid_plan = {**sample_plan, "recipes": sample_plan["recipes"][:6]}
try:
    validate_meal_plan(invalid_plan)
except ValidationError as error:
    assert "exactly 7 recipes" in str(error)
else:
    raise AssertionError("Invalid plan should raise ValidationError")

print("All helper checks passed.")

All helper checks passed.


## 5. LangChain Setup & Prompts

In [7]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Create the configured LLM after checking required settings."""
    require_configured_setting(OPENAI_API_KEY, "OPENAI_API_KEY", ".env")
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [8]:
import time
from openai import (
    APIConnectionError,
    APITimeoutError,
    AuthenticationError,
    OpenAIError,
    RateLimitError,
)


class MealPlanGenerator:
    """Generate weekly meal plans by orchestrating focused steps."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.recipe_chain = MEAL_PLAN_PROMPT | self.llm

    def build_request(self, dietary_preference: str, theme_requests: str, additional_preferences: str) -> dict:
        """Build the prompt variables sent to the language model."""
        return {
            "dietary_preference": dietary_preference,
            "theme_requests": normalize_optional_text(theme_requests, "random diverse cuisines"),
            "additional_preferences": normalize_optional_text(additional_preferences, "none"),
            "format_instructions": self.recipe_parser.get_format_instructions(),
        }

    def call_llm(self, request: dict) -> str:
        """Call the configured language model and return its text response."""
        return self.recipe_chain.invoke(request).content

    def parse_response(self, content: str) -> dict:
        """Parse the model response into a raw meal-plan dictionary."""
        return parse_json_response(content)

    def validate_plan(self, data: dict) -> MealPlan:
        """Validate the raw meal-plan dictionary with Pydantic."""
        return validate_meal_plan(data)

    @staticmethod
    def failure(error: str) -> dict:
        """Return a consistent failure response for the UI."""
        return {"success": False, "meal_plan": None, "generation_time": None, "error": error}

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none",
    ) -> dict:
        """Generate a plan by orchestrating focused generation steps."""
        start_time = time.time()
        try:
            request = self.build_request(
                dietary_preference, theme_requests, additional_preferences
            )
            result = self.call_llm(request)
            meal_plan_data = self.parse_response(result)
            meal_plan = self.validate_plan(meal_plan_data)
        except AuthenticationError as error:
            return self.failure(format_error("MealPlanGenerator.call_llm", "AuthenticationError", str(error), "OpenAI authentication", "Check OPENAI_API_KEY in .env and confirm it is active."))
        except RateLimitError as error:
            return self.failure(format_error("MealPlanGenerator.call_llm", "RateLimitError", str(error), "OpenAI request rate limit", "Wait and retry later, or reduce request frequency."))
        except APITimeoutError as error:
            return self.failure(format_error("MealPlanGenerator.call_llm", "APITimeoutError", str(error), "OpenAI request timeout", "Retry the request and check network latency."))
        except APIConnectionError as error:
            return self.failure(format_error("MealPlanGenerator.call_llm", "APIConnectionError", str(error), "OpenAI network connection", "Check internet access and the OpenAI service status."))
        except OpenAIError as error:
            return self.failure(format_error("MealPlanGenerator.call_llm", "OpenAIError", str(error), "OpenAI API call", "Review the API error details and retry if appropriate."))
        except json.JSONDecodeError as error:
            return self.failure(format_error("MealPlanGenerator.parse_response", "JSONDecodeError", error.msg, f"line {error.lineno}, column {error.colno}", "Check that the model returned valid JSON matching the requested format."))
        except KeyError as error:
            return self.failure(format_error("MealPlanGenerator.parse_response", "KeyError", str(error), "required meal-plan response field", "Check the model output against the MealPlan schema."))
        except ValidationError as error:
            details = "; ".join(
                f"{'.'.join(str(part) for part in issue['loc'])}: {issue['msg']}"
                for issue in error.errors()
            )
            return self.failure(format_error("MealPlanGenerator.validate_plan", "ValidationError", details, "MealPlan Pydantic schema", "Ensure the response has 7 unique recipes with 5 ingredients each."))
        except ValueError as error:
            return self.failure(format_error("MealPlanGenerator.parse_response", "ValueError", str(error), "model response processing", "Check that the response contains a JSON object with the required fields."))

        return {
            "success": True,
            "meal_plan": meal_plan.model_dump(),
            "generation_time": round(time.time() - start_time, 2),
            "error": None,
        }


def initialize_generator():
    """Initialize the generator and preserve setup errors for the UI."""
    try:
        return MealPlanGenerator(), None
    except (AuthenticationError, OpenAIError, EnvironmentError, ValueError) as error:
        return None, format_error(
            "initialize_generator",
            type(error).__name__,
            str(error),
            "MealPlanGenerator setup",
            "Check .env settings and installed OpenAI/LangChain dependencies.",
        )


# Initialize generator without hiding configuration errors.
generator, GENERATOR_INIT_ERROR = initialize_generator()

## 6. Email Functionality

In [9]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_email_header(meal_plan: dict) -> list[str]:
    """Format the email title section."""
    return [
        "=" * 40,
        f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})",
        "=" * 40,
        "",
    ]


def format_shopping_list_text(meal_plan: dict) -> list[str]:
    """Format the shopping list for a plain-text email."""
    lines = ["SHOPPING LIST", "-" * 20]
    lines.extend(
        f"- {item['total_quantity']} {item['name']}"
        for item in meal_plan['shopping_list']
    )
    lines.append("")
    return lines


def format_recipe_text(recipe: dict) -> list[str]:
    """Format one recipe for a plain-text email."""
    return [
        f"\n{recipe['day'].upper()} - {recipe['theme']}",
        recipe['dish_name'],
        f"Ingredients: {format_ingredients(recipe['ingredients'])}",
        f"Instructions: {recipe['instructions']}",
    ]


def format_meal_plan_email(meal_plan: dict) -> str:
    """Compose the focused email sections into one plain-text body."""
    lines = format_email_header(meal_plan)
    lines.extend(format_shopping_list_text(meal_plan))
    lines.extend(["=" * 40, "RECIPES", "=" * 40])
    for recipe in meal_plan['recipes']:
        lines.extend(format_recipe_text(recipe))
    return "\n".join(lines)


def create_meal_plan_email(recipient_email: str, meal_plan: dict, sender_email: str):
    """Build the email message without sending it."""
    message = MIMEMultipart()
    message['From'] = sender_email
    message['To'] = recipient_email
    message['Subject'] = "Your Weekly Meal Plan"
    message.attach(MIMEText(format_meal_plan_email(meal_plan), 'plain'))
    return message


def send_email_message(message, sender_email: str, password: str) -> None:
    """Send a prepared email message through Gmail SMTP."""
    with smtplib.SMTP('smtp.gmail.com', 587) as server:
        server.starttls()
        server.login(sender_email, password)
        server.send_message(message)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Orchestrate message creation and delivery for one meal plan."""
    if not EMAIL_SENDER or not EMAIL_PASSWORD:
        return {
            "success": False,
            "message": format_error(
                "send_meal_plan_email",
                "EnvironmentError",
                "missing EMAIL_SENDER or EMAIL_PASSWORD",
                ".env email configuration",
                "Add both email settings to .env and rerun the configuration cell.",
            ),
        }

    try:
        message = create_meal_plan_email(recipient_email, meal_plan, EMAIL_SENDER)
        send_email_message(message, EMAIL_SENDER, EMAIL_PASSWORD)
    except (smtplib.SMTPException, OSError) as error:
        return {
            "success": False,
            "message": format_error(
                "send_meal_plan_email",
                type(error).__name__,
                str(error),
                "Gmail SMTP connection or authentication",
                "Check SMTP access, EMAIL_SENDER, and the Gmail app password.",
            ),
        }
    except (KeyError, ValueError) as error:
        return {
            "success": False,
            "message": format_error(
                "send_meal_plan_email",
                type(error).__name__,
                str(error),
                "meal-plan email formatting",
                "Check that every recipe and ingredient contains the required fields.",
            ),
        }

    return {"success": True, "message": f"Email sent to {recipient_email}"}

## 7. Validation

In [10]:
def format_validation_errors(error: ValidationError) -> list[str]:
    """Convert Pydantic errors into actionable field messages."""
    return [
        format_error(
            "validate_plan_result",
            "ValidationError",
            issue["msg"],
            f"field {'.'.join(str(part) for part in issue['loc'])}",
            "Correct the indicated field and satisfy the MealPlan schema.",
        )
        for issue in error.errors()
    ]


def validate_plan_result(meal_plan: dict) -> list[str]:
    """Return validation errors for one raw meal-plan dictionary."""
    try:
        validate_meal_plan(meal_plan)
    except ValidationError as error:
        return format_validation_errors(error)
    except (TypeError, KeyError) as error:
        return [format_error(
            "validate_plan_result",
            type(error).__name__,
            str(error),
            "raw meal-plan value",
            "Provide a dictionary matching the MealPlan schema.",
        )]
    return []


def run_basic_validation(meal_plan_result: dict) -> dict:
    """Turn a generation result into a simple validation status."""
    if not isinstance(meal_plan_result, dict):
        return {"passed": False, "errors": [format_error(
            "run_basic_validation",
            "TypeError",
            "expected a result dictionary",
            "generation result",
            "Pass the dictionary returned by MealPlanGenerator.generate.",
        )]}
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error") or format_error(
            "run_basic_validation",
            "ValueError",
            "generation failed without an error message",
            "generation result",
            "Inspect the generator error before retrying.",
        )]}
    if "meal_plan" not in meal_plan_result:
        return {"passed": False, "errors": [format_error(
            "run_basic_validation",
            "KeyError",
            "missing meal_plan",
            "generation result",
            "Return a meal_plan field when generation succeeds.",
        )]}

    errors = validate_plan_result(meal_plan_result["meal_plan"])
    return {"passed": not errors, "errors": errors}

## 8. Gradio Interface

In [11]:
import gradio as gr
import time as pytime


def format_shopping_list_markdown(meal_plan: dict) -> str:
    """Format the shopping list for the Gradio Markdown view."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = format_ingredients(recipe['ingredients'])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def generate_plan_for_inputs(dietary_pref, theme_requests, additional_prefs):
    """Generate a plan from UI inputs after applying input defaults."""
    if generator is None:
        return {
            "success": False,
            "meal_plan": None,
            "generation_time": None,
            "error": format_error(
                "generate_plan_for_inputs",
                "ConfigurationError",
                GENERATOR_INIT_ERROR or "generator is unavailable",
                "generator initialization",
                "Fix the configuration error and rerun the setup cells.",
            ),
        }
    return generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=normalize_optional_text(theme_requests, "random"),
        additional_preferences=normalize_optional_text(additional_prefs, "none"),
    )


def format_streaming_header(meal_plan: dict) -> str:
    """Build the initial Markdown shown before recipes stream."""
    return (
        f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
        f"{format_shopping_list_markdown(meal_plan)}\n\n---\n\n"
        "## Recipes\n\n"
    )


def append_recipe_to_output(output: str, recipe: dict) -> str:
    """Append one formatted recipe to the current Markdown output."""
    return output + format_single_recipe(recipe) + "\n"


def format_generation_status(result: dict) -> str:
    """Format the final generation timing and validation status."""
    validation = run_basic_validation(result)
    status = (
        "All checks passed"
        if validation["passed"]
        else f"Issues: {', '.join(validation['errors'])}"
    )
    return f"---\n*Generated in {result['generation_time']}s | {status}*"


def validate_recipient_email(email: str) -> str | None:
    """Return an error message when an email address is invalid."""
    if not email or "@" not in email:
        return format_error(
            "validate_recipient_email",
            "ValueError",
            "invalid email address",
            "recipient email field",
            "Enter an address containing a local part, @, and a domain.",
        )
    return None


def format_generation_failure(result: dict) -> str:
    """Format a generation failure for a UI status field."""
    return f"Generation failed: {result['error']}"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Stream a generated plan while delegating each concern to helpers."""
    output = "Generating your meal plan..."
    yield output

    result = generate_plan_for_inputs(
        dietary_pref, theme_requests, additional_prefs
    )
    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    try:
        output = format_streaming_header(result["meal_plan"])
        yield output
        pytime.sleep(0.1)

        for recipe in result["meal_plan"]["recipes"]:
            output = append_recipe_to_output(output, recipe)
            yield output
            pytime.sleep(0.05)

        yield output + format_generation_status(result)
    except (KeyError, TypeError, ValueError) as error:
        yield format_error(
            "generate_plan_streaming",
            type(error).__name__,
            str(error),
            "generated plan presentation",
            "Check the generated response fields before displaying the plan.",
        )


def send_email_ui(email, dietary_pref, theme_requests, additional_prefs):
    """Validate inputs, generate a plan, and delegate email delivery."""
    validation_error = validate_recipient_email(email)
    if validation_error:
        return validation_error

    result = generate_plan_for_inputs(
        dietary_pref, theme_requests, additional_prefs
    )
    if not result["success"]:
        return format_generation_failure(result)

    return send_meal_plan_email(email, result["meal_plan"])["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, dietary_dropdown, theme_input, additional_input],
        outputs=[email_status]
    )

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Modular function checks

In [12]:
# These checks cover formatting, validation, UI input, and message construction only.
assert format_shopping_list_markdown(sample_plan).startswith("## Shopping List")
assert "ingredient-0" in format_single_recipe(sample_plan["recipes"][0])
assert "WEEKLY MEAL PLAN (VEGAN)" in format_meal_plan_email(sample_plan)
assert "invalid email address" in validate_recipient_email("bad-address")
assert validate_recipient_email("person@example.com") is None
assert format_generation_failure({"error": "test failure"}) == "Generation failed: test failure"

message = create_meal_plan_email("person@example.com", sample_plan, "sender@example.com")
assert message["To"] == "person@example.com"
assert "Pasta" in message.get_payload()[0].get_payload()

assert "All checks passed" in format_generation_status({
    "success": True,
    "meal_plan": sample_plan,
    "generation_time": 0.1,
    "error": None,
})
print("All modular function checks passed.")

All modular function checks passed.


### Error-handling checks

In [13]:
# Invalid-input checks; these do not call OpenAI or SMTP.
try:
    require_configured_setting(None, "OPENAI_API_KEY", ".env")
except EnvironmentError as error:
    assert "require_configured_setting" in str(error)
    assert "EnvironmentError" in str(error)
    assert "Suggestion:" in str(error)
else:
    raise AssertionError("Missing configuration should raise EnvironmentError")

try:
    parse_json_response("{\"recipes\": []}")
except KeyError as error:
    assert "parse_json_response" in str(error)
    assert "missing required fields" in str(error)
    assert "Suggestion:" in str(error)
else:
    raise AssertionError("Missing response fields should raise KeyError")

try:
    format_ingredients([{"name": "rice"}])
except KeyError as error:
    assert "format_ingredients" in str(error)
    assert "ingredient index 0" in str(error)
    assert "Suggestion:" in str(error)
else:
    raise AssertionError("Missing ingredient fields should raise KeyError")

invalid_result = run_basic_validation({"success": True, "meal_plan": {}})
assert invalid_result["passed"] is False
assert all("Suggestion:" in error for error in invalid_result["errors"])

mock_generator = object.__new__(MealPlanGenerator)
mock_generator.build_request = lambda *args: {}
mock_generator.call_llm = lambda request: "Model reply: {not valid json}"
generation_error = mock_generator.generate("vegan")
assert generation_error["success"] is False
assert "MealPlanGenerator.parse_response" in generation_error["error"]
assert "JSONDecodeError" in generation_error["error"]
assert "Suggestion:" in generation_error["error"]

email_error = send_meal_plan_email("person@example.com", sample_plan)
assert email_error["success"] is False
assert "send_meal_plan_email" in email_error["message"]
assert "EnvironmentError" in email_error["message"]
assert "Suggestion:" in email_error["message"]

print("All error-handling checks passed.")

All error-handling checks passed.


## 9. Refactored Code Tests

In [14]:
# Deterministic tests for valid behavior and actionable failure messages.
# The mocked API responses keep this test local and reproducible.
import os
import tempfile
import uuid

mock_generator = object.__new__(MealPlanGenerator)
mock_generator.build_request = lambda *args: {}
mock_generator.call_llm = lambda request: json.dumps(sample_plan)
valid_result = mock_generator.generate("vegan")
assert valid_result["success"] is True
assert set(valid_result) == {"success", "meal_plan", "generation_time", "error"}
assert len(valid_result["meal_plan"]["recipes"]) == 7
assert all(len(recipe["ingredients"]) == 5 for recipe in valid_result["meal_plan"]["recipes"])
assert run_basic_validation(valid_result)["passed"] is True
assert "Pasta" in format_meal_plan_email(valid_result["meal_plan"])

mock_generator.call_llm = lambda request: "Model reply: {not valid json}"
malformed_result = mock_generator.generate("vegan")
assert malformed_result["success"] is False
assert "MealPlanGenerator.parse_response" in malformed_result["error"]
assert "JSONDecodeError" in malformed_result["error"]
assert "line" in malformed_result["error"] and "column" in malformed_result["error"]
assert "Suggestion:" in malformed_result["error"]

invalid_plan_payload = {"dietary_preference": "vegan", "recipes": [], "shopping_list": []}
mock_generator.call_llm = lambda request: json.dumps(invalid_plan_payload)
invalid_plan_result = mock_generator.generate("vegan")
assert invalid_plan_result["success"] is False
assert "ValidationError" in invalid_plan_result["error"]
assert "recipes" in invalid_plan_result["error"]
assert "Suggestion:" in invalid_plan_result["error"]

invalid_field_plan = {**sample_plan, "recipes": [
    {**sample_plan["recipes"][0], "ingredients": [
        {"quantity": "1 cup"}, *sample_plan["recipes"][0]["ingredients"][1:]
    ]}, *sample_plan["recipes"][1:]
]}
field_validation = run_basic_validation({"success": True, "meal_plan": invalid_field_plan})
assert field_validation["passed"] is False
assert any("recipes.0.ingredients.0.name" in error for error in field_validation["errors"])
assert all("Suggestion:" in error for error in field_validation["errors"])

missing_path = os.path.join(tempfile.gettempdir(), f"meal_planner_missing_{uuid.uuid4().hex}.json")
try:
    load_json_file(missing_path)
except FileNotFoundError as error:
    message = str(error)
    assert "load_json_file" in message
    assert "FileNotFoundError" in message
    assert missing_path in message
    assert "Suggestion:" in message
else:
    raise AssertionError("Missing JSON file should raise FileNotFoundError")

invalid_json_path = None
try:
    with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False, encoding="utf-8") as temporary_file:
        temporary_file.write('{"recipes": [\n  }')
        invalid_json_path = temporary_file.name
    load_json_file(invalid_json_path)
except json.JSONDecodeError as error:
    message = str(error)
    assert "load_json_file" in message
    assert "line" in message and "column" in message
    assert "Suggestion:" in message
else:
    raise AssertionError("Invalid JSON should raise JSONDecodeError")
finally:
    if invalid_json_path and os.path.exists(invalid_json_path):
        os.unlink(invalid_json_path)

def raise_api_error(request):
    raise OpenAIError("simulated API failure")

mock_generator.call_llm = raise_api_error
api_error_result = mock_generator.generate("vegan")
assert api_error_result["success"] is False
assert "MealPlanGenerator.call_llm" in api_error_result["error"]
assert "OpenAIError" in api_error_result["error"]
assert "simulated API failure" in api_error_result["error"]
assert "Suggestion:" in api_error_result["error"]

print("All refactored-code tests passed.")

All refactored-code tests passed.


## 10. Launch Application

In [15]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://eaaeb70f3c42f49f08.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 11. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [16]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

omnivore -> PASS
vegetarian -> PASS
vegan -> PASS
keto -> PASS
